# Hybrid model training on Colab

This notebook executes repository scripts; it is not a substitute for an executed training report. Select a GPU runtime, place a project checkout at `/content/healthier`, and retain the source dataset and manifests. All data preparation below uses public source records. Optional conditional-severity predictions apply only to documented source records with unknown severity and remain separate from source labels.

The text and image robustness experiments are separate from held-out source-severity evaluation. The model does not predict interaction existence for unrecorded pairs. This notebook does not provide clinical validation.

In [ ]:
from pathlib import Path
import subprocess, sys, json
REPO_DIR = Path('/content/healthier')
if not (REPO_DIR / 'scripts/train_ml.py').is_file():
    raise FileNotFoundError('Place the complete project checkout at /content/healthier first.')
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=REPO_DIR, check=True)
run('-m', 'pip', 'install', '-r', 'requirements-ml.txt')
import torch
print({'torch': torch.__version__, 'cuda_available': torch.cuda.is_available(),
       'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})

## Public molecule cache and source-label preparation

Molecule resolution is resumable and may take substantial time. Keep `artifacts/ml/molecules.json` between runs. Do not replace missing structures with fabricated molecules. Source identifiers, join coverage and hashes are recorded by the preparation scripts.

In [ ]:
source_csv = REPO_DIR / 'data/processed_ddinter/ddinter_cleaned.csv'
if not source_csv.is_file():
    raise FileNotFoundError('Include the processed DDInter source CSV produced by the repository preprocessing script.')
run('scripts/prepare_molecules.py', '--output', 'artifacts/ml/molecules.json', '--workers', '4')
run('scripts/prepare_ml_dataset.py', '--source', source_csv,
    '--molecules', 'artifacts/ml/molecules.json', '--output', 'artifacts/ml/dataset')

## Train the registered model and split comparisons

Run pair and cold-drug protocols with all requested seeds. If a run fails, retain its failure and resume according to the trainer's CLI; do not mark full training completed from a smoke run. Check `python scripts/train_ml.py --help` for optional runtime controls in the checked-out revision.

In [ ]:
run('scripts/train_ml.py', '--protocols', 'pair', 'cold',
    '--models', 'fingerprint', 'graphsage', 'fusion', 'role_fusion',
    '--seeds', '17', '29', '43', '--device', 'cuda' if torch.cuda.is_available() else 'cpu')

## Inspect and preserve results

Review held-out metrics, calibration, cold-drug performance, leakage audits and dataset coverage before selecting a deployment artifact. These metrics describe source labels; they do not estimate individual patient outcomes.

In [ ]:
ml_root = REPO_DIR / 'artifacts/ml'
reports = sorted(ml_root.rglob('summary.json')) + sorted(ml_root.rglob('metrics.json'))
for path in reports:
    print(path.relative_to(REPO_DIR))
    report = json.loads(path.read_text())
    print(json.dumps(report, indent=2)[:12000])
if not reports:
    raise RuntimeError('No executed training metrics found; inspect trainer output before claiming completion.')
import shutil
archive = shutil.make_archive('/content/hybrid_model_export', 'zip', root_dir=ml_root)
print('Saved model artifacts and their original manifests:', archive)
# Optional in Colab: from google.colab import files; files.download(archive)